In [20]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [3]:
spark = SparkSession.builder.appName("Feature_Engineering").getOrCreate()

In [4]:
data_folder = "./sales_forecasting_data"
df = spark.read.csv(f"{data_folder}/cleaned_data", header=True, inferSchema=True)

In [ ]:
df.show()

+----------+---------+--------+--------+----------+-----------+---------+-----+----------+-----+---------+----+-------+------------+----------+-------+
|      date|store_nbr|item_nbr|      id|unit_sales|onpromotion|   family|class|perishable| city|    state|type|cluster|transactions|dcoilwtico|holiday|
+----------+---------+--------+--------+----------+-----------+---------+-----+----------+-----+---------+----+-------+------------+----------+-------+
|2016-12-04|       48| 1935388|98928753|       2.0|      false|   BEAUTY| 4255|         0|Quito|Pichincha|   A|     14|        4427|      51.7|  false|
|2016-12-04|       48| 1936589|98928754|       3.0|      false|GROCERY I| 1034|         0|Quito|Pichincha|   A|     14|        4427|      51.7|  false|
|2016-12-04|       48| 1936931|98928755|      22.0|       true|BEVERAGES| 1124|         0|Quito|Pichincha|   A|     14|        4427|      51.7|  false|
|2016-12-04|       48| 1937039|98928756|      25.0|       true|BEVERAGES| 1124|         

In [5]:
df.orderBy("date").show()

+----------+---------+--------+---+----------+-----------+-------------+-----+----------+-------+-----------+----+-------+------------+----------+-------+
|      date|store_nbr|item_nbr| id|unit_sales|onpromotion|       family|class|perishable|   city|      state|type|cluster|transactions|dcoilwtico|holiday|
+----------+---------+--------+---+----------+-----------+-------------+-----+----------+-------+-----------+----+-------+------------+----------+-------+
|2013-01-01|       25|  103665|  0|       7.0|      false| BREAD/BAKERY| 2712|         1|Salinas|Santa Elena|   D|      1|         770|     93.14|   true|
|2013-01-01|       25|  105574|  1|       1.0|      false|    GROCERY I| 1045|         0|Salinas|Santa Elena|   D|      1|         770|     93.14|   true|
|2013-01-01|       25|  105575|  2|       2.0|      false|    GROCERY I| 1045|         0|Salinas|Santa Elena|   D|      1|         770|     93.14|   true|
|2013-01-01|       25|  108079|  3|       1.0|      false|    GROCERY 

Now, for the feature engineering, I started with creating day of week, month, and day of month since we found them to be relavant during the EDA.

day of week --> Sales are higher during weekends

month --> Sales differ quite a bit each month, with December having the highest sales

day of month --> Sales are higher at the start of month

In [6]:
# Add features
df = df.withColumn("day_of_week", F.date_format(F.col("date"), "EEEE"))
df = df.withColumn("month", F.date_format(F.col("date"), "MMMM"))
df = df.withColumn("day_of_month", F.dayofmonth(F.col("date")))

df = df.withColumn("day_of_week", F.dayofweek(F.col("date"))) # 1=Sunday, 2=Monday, ..., 7=Saturday
df = df.withColumn("month", F.month(F.col("date"))) 

Next is the promotion count features, which we found from the EDA as well that it has one of the highest impact, so I create a count for 1-day, 1-week, 2-week and 1-month.

1-day because promotion on previous day could impact today's sale.

1-week and 2-week means it's a weekly timeframe, and include every day of week.

In [7]:
# Convert T/F to 1/0
df = df.withColumn("onpromotion", F.col("onpromotion").cast("int"))

# Define windows
window_7 = Window.partitionBy("store_nbr", "item_nbr").orderBy("date").rowsBetween(-6, 0)
window_14 = Window.partitionBy("store_nbr", "item_nbr").orderBy("date").rowsBetween(-13, 0)
window_30 = Window.partitionBy("store_nbr", "item_nbr").orderBy("date").rowsBetween(-29, 0)

# Add features
df = df.withColumn("promo_last_7_days", F.sum("onpromotion").over(window_7))
df = df.withColumn("promo_last_14_days", F.sum("onpromotion").over(window_14))
df = df.withColumn("promo_last_30_days", F.sum("onpromotion").over(window_30))

Similarly, I create sales lag for the mentioned timeframe.

Because there are different types of items, some like eggs is going to always sell well, whereas things like furtinure is going to have lower sales, so past sales is a good indicator.

In [8]:
window_spec = Window.partitionBy("store_nbr", "item_nbr").orderBy("date")

df = df.withColumn("sales_lag_1", F.lag("unit_sales", 1).over(window_spec))
df = df.withColumn("sales_lag_7", F.lag("unit_sales", 7).over(window_spec))
df = df.withColumn("sales_lag_14", F.lag("unit_sales", 14).over(window_spec))
df = df.withColumn("sales_lag_30", F.lag("unit_sales", 30).over(window_spec))

df = df.withColumn("sales_lag_21", F.lag("unit_sales", 21).over(window_spec))
df = df.withColumn("sales_lag_28", F.lag("unit_sales", 28).over(window_spec))

df = df.withColumn("weekly_sales_avg", 
                   (F.col("sales_lag_7") + F.col("sales_lag_14") + F.col("sales_lag_21") + F.col("sales_lag_28")) / 4)
df = df.drop("sales_lag_21", "sales_lag_28")

Also I created a weekly sales average which is just the average of the last 4 day of week.

Next, I create rolling averages for the same timeframe.

In [ ]:
window_7 = Window.partitionBy("store_nbr", "item_nbr").orderBy("date").rowsBetween(-7, -1)
window_14 = Window.partitionBy("store_nbr", "item_nbr").orderBy("date").rowsBetween(-14, -1)
window_30 = Window.partitionBy("store_nbr", "item_nbr").orderBy("date").rowsBetween(-30, -1)


df = df.withColumn("rolling_7_mean", F.avg("unit_sales").over(window_7))
df = df.withColumn("rolling_14_mean", F.avg("unit_sales").over(window_14))
df = df.withColumn("rolling_30_mean", F.avg("unit_sales").over(window_30))

I also create a feature for perishable item, using a 1-week timeframe, since most people do their grocery weekly.

In [10]:
df = df.withColumn("perishable_rolling_7_mean", F.col("rolling_7_mean") * F.col("perishable"))

In [11]:
df.show()

25/08/31 10:23:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+---------+--------+--------+----------+-----------+---------+-----+----------+-----+---------+----+-------+------------+----------+-------+-----------+-----+------------+-----------------+------------------+------------------+-----------+-----------+------------+------------+----------------+------------------+------------------+------------------+-------------------------+
|      date|store_nbr|item_nbr|      id|unit_sales|onpromotion|   family|class|perishable| city|    state|type|cluster|transactions|dcoilwtico|holiday|day_of_week|month|day_of_month|promo_last_7_days|promo_last_14_days|promo_last_30_days|sales_lag_1|sales_lag_7|sales_lag_14|sales_lag_30|weekly_sales_avg|    rolling_7_mean|   rolling_14_mean|   rolling_30_mean|perishable_rolling_7_mean|
+----------+---------+--------+--------+----------+-----------+---------+-----+----------+-----+---------+----+-------+------------+----------+-------+-----------+-----+------------+-----------------+------------------+---

In [12]:
df = df.fillna(0)

There are missing values at the start of the data. Since it's only the starting values that I don't know, I'm going just fill them with 0.

In [13]:
df.show()

+----------+---------+--------+--------+----------+-----------+---------+-----+----------+-----+---------+----+-------+------------+----------+-------+-----------+-----+------------+-----------------+------------------+------------------+-----------+-----------+------------+------------+----------------+------------------+------------------+------------------+-------------------------+
|      date|store_nbr|item_nbr|      id|unit_sales|onpromotion|   family|class|perishable| city|    state|type|cluster|transactions|dcoilwtico|holiday|day_of_week|month|day_of_month|promo_last_7_days|promo_last_14_days|promo_last_30_days|sales_lag_1|sales_lag_7|sales_lag_14|sales_lag_30|weekly_sales_avg|    rolling_7_mean|   rolling_14_mean|   rolling_30_mean|perishable_rolling_7_mean|
+----------+---------+--------+--------+----------+-----------+---------+-----+----------+-----+---------+----+-------+------------+----------+-------+-----------+-----+------------+-----------------+------------------+---

In [14]:
# Convert holiday T/F to 1/0
df = df.withColumn("holiday", F.col("holiday").cast("int"))

Now, I split the data for training and testing, using the same timeframe that I did in the EDA to avoid data leakage.

In [15]:
train_df = df.filter(F.col("date") < '2017-01-01')
test_df = df.filter(F.col("date") >= '2017-01-01')

In [ ]:
train_df.show()

25/08/31 10:02:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+---------+--------+--------+----------+-----------+---------+-----+----------+-----+---------+----+-------+------------+----------+-------+-----------+-----+------------+-----------------+------------------+------------------+-----------+-----------+------------+------------+----------------+------------------+------------------+------------------+-------------------------+
|      date|store_nbr|item_nbr|      id|unit_sales|onpromotion|   family|class|perishable| city|    state|type|cluster|transactions|dcoilwtico|holiday|day_of_week|month|day_of_month|promo_last_7_days|promo_last_14_days|promo_last_30_days|sales_lag_1|sales_lag_7|sales_lag_14|sales_lag_30|weekly_sales_avg|    rolling_7_mean|   rolling_14_mean|   rolling_30_mean|perishable_rolling_7_mean|
+----------+---------+--------+--------+----------+-----------+---------+-----+----------+-----+---------+----+-------+------------+----------+-------+-----------+-----+------------+-----------------+------------------+---

In [ ]:
train_df.coalesce(1).write.csv("train_folder", header=True, mode="overwrite")


In [ ]:
test_df.coalesce(1).write.csv("test_folder2", header=True, mode="overwrite")

In [ ]:
spark.stop()